# Framework diagrams (Graphviz) for the paper
Three diagrams of the current framework, in the same Graphviz style as before:
1. **framework_diagram**: setup once (safe, verifiable view set) · cloud (learning on real recorded views) · edge (online inspection loop on the Jetson + myCobot)
2. **transformer_diagram**: the view-selection policy (tokens, embeddings, attention, residuals, pointer decoder, action mask, REINFORCE)
3. **parta_diagram**: from one taught pose to the safe candidate set (reference advisor, lattice, checks, outputs)

Outputs: PNG (300 dpi), PDF (vector, for LaTeX) and SVG. Edit any label in cell 2 and run it again.

In [ ]:
!pip -q install graphviz
!which dot || apt-get -qq install -y graphviz > /dev/null

In [ ]:
# ===================== Framework diagrams (Graphviz, Colab) =====================
# In Colab:  !pip -q install graphviz   (the `dot` program is preinstalled; if not: !apt-get -qq install graphviz)
from graphviz import Digraph
try:
    from IPython.display import display, Image
except ImportError:
    display = Image = None

FONT = 'Helvetica'


def show(path):
    if display:
        display(Image(filename=path))


# ---------------------------------------------------------------- 1) overall framework (cloud + edge)
framework = Digraph('Framework', format='png')
framework.attr(rankdir='TB', size='14,10', dpi='300', fontname=FONT, nodesep='0.45', ranksep='0.32', newrank='true')
framework.attr('node', shape='box', style='rounded,filled', fontname=FONT, fontsize='11', margin='0.18,0.08', width='3.6')
framework.attr('edge', fontname=FONT, fontsize='10', color='#333333', penwidth='1.3')

with framework.subgraph(name='cluster_setup') as s:
    s.attr(label='SETUP ONCE (Jetson Nano): safe, verifiable view set', style='dashed,rounded', color='#2e7d32',
           fontcolor='#2e7d32', fontsize='14', fontname='Helvetica-Bold', bgcolor='#f1f8f1')
    s.node('S1', 'One taught reference pose\n(survey: whole box + markers)', fillcolor='#ffffff')
    s.node('S2', 'Reference advisor\n(reachable share 44% → 100%)', fillcolor='#ffffff')
    s.node('S3', 'Reachability lattice\nx, y  ×  dz 0…−30 mm  ×  pitch ±10°', fillcolor='#ffffff')
    s.node('S4', 'Safety filters\nIK continuation · κ ≥ 0.04 · joint margin\nfence · verifiable view', fillcolor='#ffffff')
    s.node('S5', 'Safe candidate set 𝒱\n(K views + safe route times)', fillcolor='#e6f4e6', penwidth='2')
    s.edge('S1', 'S2'); s.edge('S2', 'S3'); s.edge('S3', 'S4'); s.edge('S4', 'S5')

with framework.subgraph(name='cluster_edge') as e:
    e.attr(label='EDGE (Jetson Nano 4 GB + myCobot 280): online inspection loop', style='dashed,rounded',
           color='#ff7f0e', fontcolor='#ff7f0e', fontsize='14', fontname='Helvetica-Bold', bgcolor='#fff5e6')
    e.node('E1', 'Survey image at the taught pose\n(640 × 480)', fillcolor='#ffffff')
    e.node('E2', 'State sₜ\n(survey features, views taken, time left, pose)', fillcolor='#ffffff')
    e.node('E3', 'Policy inference πθ\n(NumPy, no PyTorch)', fillcolor='#ffffff')
    e.node('E4', 'Action mask\n(only views in 𝒱, not visited, within budget)', fillcolor='#ffffff')
    e.node('E5', 'Safe lattice route\n(bounded joint steps)', fillcolor='#ffffff')
    e.node('E6', 'myCobot 280\nexecution', fillcolor='#ffffff')
    e.node('E7', 'View verification\n(marker servo / registration: 6.2 → 0.8 mm)', fillcolor='#ffffff')
    e.node('E8', 'Capture & score\nreward rₜ = gain in inspection quality', fillcolor='#ffffff')
    e.node('E9', 'Inspection output\n(verified views, mosaic, defects)', fillcolor='#aaffaa', penwidth='2')
    e.edge('E1', 'E2'); e.edge('E2', 'E3'); e.edge('E3', 'E4'); e.edge('E4', 'E5'); e.edge('E5', 'E6')
    e.edge('E6', 'E7'); e.edge('E7', 'E8'); e.edge('E8', 'E9', label=' budget used ')
    e.edge('E8', 'E2', label=' next step ', style='dashed', constraint='false')

with framework.subgraph(name='cluster_cloud') as c:
    c.attr(label='CLOUD (Google Colab): learning on real recorded views', style='dashed,rounded', color='#1f77b4',
           fontcolor='#1f77b4', fontsize='14', fontname='Helvetica-Bold', bgcolor='#f0f8ff')
    c.node('C1', 'View library 𝒟\n(N placements × K verified real views)', fillcolor='#ffffff')
    c.node('C2', 'Patch quality scoring\nresolution × sharpness × (1 − glare)', fillcolor='#ffffff')
    c.node('C3', 'Replay environment\n(real images, exact oracle, time budget)', fillcolor='#ffffff')
    c.node('C4', 'Transformer pointer policy\n(REINFORCE, PyTorch, 3 seeds)', fillcolor='#ffffff')
    c.node('C5', 'Held-out evaluation\nvs MLP · greedy · set cover · raster · oracle', fillcolor='#ffffff')
    c.node('C6', 'Export weights (.npz)\nNumPy parity < 1e-4', fillcolor='#ffebcd', penwidth='2')
    c.edge('C1', 'C2'); c.edge('C2', 'C3'); c.edge('C3', 'C4'); c.edge('C4', 'C5'); c.edge('C5', 'C6')

# three columns side by side, aligned at the top
framework.body.append('{rank=same; S1; C1; E1;}')
framework.edge('E7', 'C1', label=' every safe view recorded\n and verified on the robot ', color='#2e7d32', penwidth='2',
               fontcolor='#2e7d32', constraint='false')
framework.edge('S5', 'C1', label=' defines the\n candidate views ', color='#2e7d32', penwidth='2', style='dashed',
               fontcolor='#2e7d32', constraint='false')
framework.edge('C6', 'E3', label=' learned weights θ\n (zero-PyTorch) ', color='#d62728', penwidth='3', style='dashed',
               fontcolor='#d62728', constraint='false')
framework.render('framework_diagram', view=False)
show('framework_diagram.png')


# ---------------------------------------------------------------- 2) transformer view-selection policy (bottom to top)
transformer = Digraph('Transformer', format='png')
transformer.attr(rankdir='BT', size='8,12', dpi='300', fontname=FONT, ranksep='0.35')
transformer.attr('node', shape='box', style='rounded,filled', fontname=FONT, fontsize='11', width='5')
transformer.attr('edge', color='#333333', penwidth='1.5', fontname=FONT, fontsize='10')

transformer.node('In', 'Input state\nK candidate-view tokens (14 features each) + 1 context token (8 features)',
                 fillcolor='#cce6ff', penwidth='2')
transformer.node('Emb', 'Linear embeddings  (14 → 48,  8 → 48)\nposition is a token feature: no positional encoding',
                 fillcolor='#ffffff')
with transformer.subgraph(name='cluster_enc') as enc:
    enc.attr(label='Encoder layer  (× 2)', style='rounded', color='#7e57c2', penwidth='2', fontname='Helvetica-Bold',
             fontcolor='#7e57c2', fontsize='12')
    with enc.subgraph(name='cluster_attn') as a:
        a.attr(label='Self-attention across all candidate views', style='rounded', color='#d62728', penwidth='2',
               fontname=FONT, fontcolor='#d62728', fontsize='10')
        a.node('QKV', 'Linear projections\nQ, K, V  (48 → 48)', fillcolor='#ffe6e6')
        a.node('Dot', 'Scaled dot-product\nSoftmax(QKᵀ / √48)', fillcolor='#ffe6e6')
        a.node('Sum', 'Weighted sum of V', fillcolor='#ffe6e6')
        a.edge('QKV', 'Dot'); a.edge('Dot', 'Sum')
    enc.node('LN1', 'Add & Layer Norm', fillcolor='#ffffff')
    enc.node('FF', 'Feed-forward network\n(48 → 96 → 48, GELU)', fillcolor='#ffffff')
    enc.node('LN2', 'Add & Layer Norm', fillcolor='#ffffff')
transformer.node('Dec', 'Pointer decoder\nquery = context token, keys = view tokens\nℓₖ = 10 · tanh(q · kₖ / √48)',
                 fillcolor='#f3e5f5')
transformer.node('Mask', 'Action mask\n(not in 𝒱 / visited / over budget → −∞)', fillcolor='#ffebcd')
transformer.node('Out', 'Softmax over the K views\n→ next view aₜ  (cell x, y · height dz · tilt)', fillcolor='#aaffaa',
                 penwidth='2')
transformer.node('RL', 'Training: REINFORCE\n∇J = E[(R − R̄) ∇ log πθ]\nR = quality gain on real recorded views,\nR̄ = mean of 8 rollouts',
                 fillcolor='#fff8dc', shape='note', width='3')

transformer.edge('In', 'Emb'); transformer.edge('Emb', 'QKV'); transformer.edge('Sum', 'LN1'); transformer.edge('LN1', 'FF')
transformer.edge('FF', 'LN2'); transformer.edge('LN2', 'Dec'); transformer.edge('Dec', 'Mask'); transformer.edge('Mask', 'Out')
transformer.edge('Emb', 'LN1', style='dashed', color='#d62728', label=' Residual')
transformer.edge('LN1', 'LN2', style='dashed', color='#d62728', label=' Residual')
transformer.edge('RL', 'Dec', style='dotted', color='#b8860b', label=' policy gradient ', constraint='false')
transformer.render('transformer_diagram', view=False)
show('transformer_diagram.png')


# ---------------------------------------------------------------- 3) Part A: from one taught pose to the safe view set
parta = Digraph('PartA', format='png')
parta.attr(rankdir='TB', size='9,11', dpi='300', fontname=FONT, ranksep='0.35', nodesep='0.35')
parta.attr('node', shape='box', style='rounded,filled', fontname=FONT, fontsize='11', width='4.2')
parta.attr('edge', color='#333333', penwidth='1.4', fontname=FONT, fontsize='10')

with parta.subgraph(name='cluster_in') as i:
    i.attr(label='Inputs', style='dashed,rounded', color='#1f77b4', fontcolor='#1f77b4', fontname='Helvetica-Bold')
    i.node('P1', '<Taught reference pose q<SUB>ref</SUB>>', fillcolor='#cce6ff', width='2.6')
    i.node('P2', 'Calibration\n(image Jacobian A, markers,\ntable grid, camera height H)', fillcolor='#cce6ff', width='2.6')
parta.node('ADV', 'Reference advisor\nmove box + camera toward the base until the\nwhole box is reachable (44% → 100%)', fillcolor='#ffebcd',
           penwidth='2')
parta.node('LAT', '<Reachability lattice around q<SUB>ref</SUB><BR/>14 × 14 offsets (10 mm) × heights 0…−30 mm × pitch {0, ±10°}>',
           fillcolor='#ffffff')
with parta.subgraph(name='cluster_chk') as k:
    k.attr(label='Checks for every lattice pose', style='rounded', color='#d62728', penwidth='2', fontcolor='#d62728',
           fontname='Helvetica-Bold')
    k.node('K1', 'IK by continuation from a solved neighbour\n(≤ 25° joint change: one joint branch)', fillcolor='#ffe6e6')
    k.node('K2', '<Conditioning κ = σ<SUB>min</SUB> / σ<SUB>max</SUB> ≥ 0.04<BR/>(away from singularities)>', fillcolor='#ffe6e6')
    k.node('K3', 'Joint-limit margin ≥ 5°  ·  inside the fence (drawn box)', fillcolor='#ffe6e6')
    k.node('K4', 'Verifiable: markers in view or registration to the survey,\nlens in focus, ≥ 25 mm above the part', fillcolor='#ffe6e6')
    k.edge('K1', 'K2'); k.edge('K2', 'K3'); k.edge('K3', 'K4')
parta.node('REJ', 'rejected pose\n(not offered to any planner)', fillcolor='#eeeeee', fontcolor='#777777', width='2.4')
parta.node('V', 'Safe candidate set 𝒱\nK views + safe route time between every pair', fillcolor='#aaffaa', penwidth='2')
parta.node('CP', 'Classical baseline\nweighted set cover + Dijkstra routes', fillcolor='#ffffff', width='3')
parta.node('RLA', 'Action space of the learned policy\n(Part B, RL)', fillcolor='#f3e5f5', width='3', penwidth='2')
parta.edge('P1', 'ADV'); parta.edge('P2', 'ADV'); parta.edge('ADV', 'LAT'); parta.edge('LAT', 'K1'); parta.edge('K4', 'V', label=' pass ')
for kk in ('K1', 'K2', 'K3', 'K4'):
    parta.edge(kk, 'REJ', style='dashed', color='#999999', arrowsize='0.6')
parta.edge('V', 'CP'); parta.edge('V', 'RLA')
parta.render('parta_diagram', view=False)
show('parta_diagram.png')


# ---------------------------------------------------------------- vector versions for LaTeX (PDF) and editing (SVG)
for g, name in ((framework, 'framework_diagram'), (transformer, 'transformer_diagram'), (parta, 'parta_diagram')):
    for fmt in ('pdf', 'svg'):
        g.format = fmt
        g.render(name, view=False)
print('saved framework_diagram, transformer_diagram, parta_diagram as .png (300 dpi), .pdf and .svg')


In [ ]:
import zipfile, glob
from google.colab import files
with zipfile.ZipFile('framework_diagrams.zip', 'w') as z:
    for f in glob.glob('*_diagram.*'):
        if f.split('.')[-1] in ('png', 'pdf', 'svg'):
            z.write(f)
files.download('framework_diagrams.zip')